In [0]:
!nc -vz pub.worldb.dedyn.io 9003

DNS fwd/rev mismatch: pub.worldb.dedyn.io != 191.248.168.241.dynamic.adsl.gvt.net.br
pub.worldb.dedyn.io [191.248.168.241] 9003 (?) open


# Databricks Free Edition doesn't include the Trino JDBC driver

In [0]:
trino_host = "pub.worldb.dedyn.io"
trino_port = 9003

trino_user = "my-user"
trino_password = "my-password"

catalog = "postgresql"
schema = "public"

jdbc_url = (
    f"jdbc:trino://{trino_host}:{trino_port}/{catalog}/{schema}"
    "?SSL=true"
    "&SSLVerification=FULL"
)

df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("driver", "io.trino.jdbc.TrinoDriver")
    .option("user", trino_user)
    .option("password", trino_password)
    .option("query", "SELECT * FROM my_table LIMIT 10")
    .load()
)

#display(df)

# Workaround: use the Trino Python client

In [0]:
%pip install trino cryptography

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!ls -la /Workspace/Users/rogermm@gmail.com/datahub/files/docker/trino/certificates/ca.crt
!ls -la /Workspace/Users/rogermm@gmail.com/datahub/files/docker/trino/certificates/trino-client.p12

-rwxrwxrwx 1 root root 1452 Aug  7 19:47 /Workspace/Users/rogermm@gmail.com/datahub/files/docker/trino/certificates/ca.crt
-rwxrwxrwx 1 root root 5012 Aug  7 19:47 /Workspace/Users/rogermm@gmail.com/datahub/files/docker/trino/certificates/trino-client.p12


In [0]:
for scope in dbutils.secrets.listScopes():
    print(f"\nScope: {scope.name}")
    for secret in dbutils.secrets.list(scope.name):
        print(f"  {secret.key}")


Scope: on-premises-integration
  trino-client-keystore-password

Scope: postgres_credentials
  xxx


In [0]:
dbutils.secrets.list("on-premises-integration")

[SecretMetadata(key='trino-client-keystore-password')]

In [0]:
def prepare_trino_mtls(client_keystore_path, client_keystore_password):
    private_key, client_certificate, _ = pkcs12.load_key_and_certificates(
        client_keystore_path.read_bytes(),
        client_keystore_password,
    )

    if private_key is None or client_certificate is None:
        raise ValueError(
            "trino-client.p12 does not contain a client identity"
        )

    temporary_dir = tempfile.TemporaryDirectory(prefix="trino-mtls-")
    temporary_path = Path(temporary_dir.name)

    client_certificate_path = temporary_path / "client.crt"
    client_key_path = temporary_path / "client.key"

    client_certificate_path.write_bytes(
        client_certificate.public_bytes(Encoding.PEM)
    )

    client_key_path.write_bytes(
        private_key.private_bytes(
            Encoding.PEM,
            PrivateFormat.PKCS8,
            NoEncryption(),
        )
    )

    os.chmod(client_key_path, 0o600)

    return (
        temporary_dir,
        client_certificate_path,
        client_key_path,
    )


ca_path = Path(
    "/Workspace/Users/rogermm@gmail.com/datahub/files/docker/trino/"
    "certificates/ca.crt"
)

client_keystore_path = Path(
    "/Workspace/Users/rogermm@gmail.com/datahub/files/docker/trino/"
    "certificates/trino-client.p12"
)

client_keystore_password = dbutils.secrets.get(
    scope="on-premises-integration",
    key="trino-client-keystore-password",
).encode()


temporary_dir, client_certificate_path, client_key_path = prepare_trino_mtls(
    client_keystore_path,
    client_keystore_password,
)

In [0]:
conn = trino.dbapi.connect(
    host="pub.worldb.dedyn.io",
    port=9003,
    user="trino-client",
    catalog="postgresql",
    schema="public",
    http_scheme="https",
    auth=CertificateAuthentication(
        str(client_certificate_path),
        str(client_key_path),
    ),
    verify=str(ca_path),
)

In [0]:
cursor = conn.cursor()

cursor.execute("SELECT * FROM system.runtime.queries")

rows = cursor.fetchall()

display(rows)

_1,_2,_3,_4,_5,_6,_7,_8,_9,_10,_11,_12,_13,_14,_15
20260807_204602_00011_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),1,10,19,2026-08-07T20:46:02.739Z,2026-08-07T20:46:02.751Z,2026-08-07T20:46:03.127Z,2026-08-07T20:46:03.297Z,null,null
20260807_204915_00013_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),1,8,14,2026-08-07T20:49:15.949Z,2026-08-07T20:49:15.958Z,2026-08-07T20:49:16.306Z,2026-08-07T20:49:16.445Z,null,null
20260807_204642_00012_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),1,11,19,2026-08-07T20:46:42.332Z,2026-08-07T20:46:42.344Z,2026-08-07T20:46:42.923Z,2026-08-07T20:46:42.923Z,null,null
20260807_203747_00004_ufqft,FINISHED,trino-client,trino-python-client,SELECT version(),List(global),2,35,75,2026-08-07T20:37:47.899Z,2026-08-07T20:37:47.935Z,2026-08-07T20:37:48.294Z,2026-08-07T20:37:48.370Z,null,null
20260807_204950_00014_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),1,7,14,2026-08-07T20:49:50.695Z,2026-08-07T20:49:50.703Z,2026-08-07T20:49:51.250Z,2026-08-07T20:49:51.250Z,null,null
20260807_204956_00015_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),1,7,14,2026-08-07T20:49:56.822Z,2026-08-07T20:49:56.830Z,2026-08-07T20:49:57.199Z,2026-08-07T20:49:57.271Z,null,null
20260807_203618_00002_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),1,11,25,2026-08-07T20:36:18.796Z,2026-08-07T20:36:18.808Z,2026-08-07T20:36:19.172Z,2026-08-07T20:36:19.306Z,null,null
20260807_204534_00009_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),2,12,21,2026-08-07T20:45:34.805Z,2026-08-07T20:45:34.819Z,2026-08-07T20:45:35.197Z,2026-08-07T20:45:35.278Z,null,null
20260807_203733_00003_ufqft,FAILED,trino-client,trino-python-client,SELECT version();,null,0,0,0,2026-08-07T20:37:33.631Z,2026-08-07T20:37:33.631Z,2026-08-07T20:37:33.631Z,2026-08-07T20:37:33.631Z,USER_ERROR,SYNTAX_ERROR
20260807_203605_00001_ufqft,FINISHED,trino-client,trino-python-client,SELECT * FROM system.runtime.queries,List(global),1,10,22,2026-08-07T20:36:05.373Z,2026-08-07T20:36:05.384Z,2026-08-07T20:36:05.730Z,2026-08-07T20:36:05.895Z,null,null


In [0]:
conn.close()
temporary_dir.cleanup()